In [ ]:
import os
os.makedirs('./model', exist_ok=True)

In [ ]:
%%writefile teacher_trainer.py

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader
from evaluation import mc_cal


def train_teacher(model, config, known_unknown=0, unknown_unknown=1):

    M = config['M']
    R = config['R']
    T = config['T']
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))  # mean, std of MNIST
    ])

    allowed_indiced = []
    for i in range(10):
        if i not in [known_unknown, unknown_unknown]:
            allowed_indiced.append(i)

    batch_size = 64
    learning_rate = 0.001
    num_epochs = 30
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
    test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

    def filter_digits(dataset, exclude):
        keep_indices = [i for i, (_, label) in enumerate(dataset) if label not in exclude]
        return Subset(dataset, keep_indices)

    train_dataset = filter_digits(train_dataset, exclude=[known_unknown, unknown_unknown])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    schedule = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.25)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for data, target in train_loader:
            index = torch.randint(0, M, [len(target)])
            
            data, target = data.to(device), target.to(device)

            for i in range(len(target)):
                target[i] = allowed_indiced.index(target[i])
            
            optimizer.zero_grad()
            output = model(data, index)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch+1) % 5 ==0:
            print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}")
        schedule.step()

    model.train()
    with torch.no_grad():
        result = mc_cal(model, test_loader, allowed_indiced, T=T, M=M, R=R)
        print(result)

    torch.save(model.state_dict(), "./model/teacher_model.pth")


In [ ]:
%%writefile teacher.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class ControlledDropout(nn.Module):
    def __init__(self, p, shape, device, M=10, rw=.5):
        super(ControlledDropout, self).__init__()
        self.store = []
        for i in range(M):
            mask = torch.empty(shape, device=device).bernoulli_(1 - p)
            self.store.append(mask)

        self.rw = rw
        self.p = p

    def forward(self, x, index):
        selected = torch.stack([self.store[i] for i in index], dim=0)
        # random_walk = torch.empty_like(selected, device=x.device).bernoulli_(1-self.rw)
        random_walk = torch.ones_like(selected, device=x.device) * self.rw
        random_walk = random_walk + selected * (2*self.p-1)/(1-self.p) * self.rw
        random_walk = torch.bernoulli(random_walk)
        
        mask = (selected.int() ^ random_walk.int()) / (1 - self.p)
        # print(mask.mean(), selected.mean(), random_walk.mean())
        # mask = (selected.int()) / (1-self.p)

        return x * mask

    def get_candidates(self, indices):
        return [self.store[i] for i in indices]




class Dropout(nn.Module):
    def __init__(self, p):
        super(Dropout, self).__init__()
        
        self.p = p

    def forward(self, x):

        mask = torch.empty_like(x).bernoulli_(1 - self.p) / (1 - self.p)

        return x * mask



class ConvNet(nn.Module):
    def __init__(self, cfg):
        super(ConvNet, self).__init__()


        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False)
        self.drop1 = ControlledDropout(cfg['p'], [32, 14, 14], cfg['device'], M=cfg['M'], rw=cfg['rw'])

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.drop2 = ControlledDropout(cfg['p'], [64, 7, 7], cfg['device'], M=cfg['M'], rw=cfg['rw'])


        self.pool = nn.MaxPool2d(2, 2)  # halves spatial size


        self.fc1 = nn.Linear(64 * 7 * 7, 256)  # after pooling twice: 32→16→8
        self.fc2 = nn.Linear(256, cfg['num_classes'])


    def forward(self, x, index):

        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = self.drop1(x, index)

        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = self.drop2(x, index)

        x = x.view(x.size(0), -1)
        # x = self.drop3(x)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


    def get_candidates(self, indices):
        mask1 = self.drop1.get_candidates(indices)
        mask2 = self.drop2.get_candidates(indices)

        return [mask1, mask2]




class ConvNet_Simple(nn.Module):
    def __init__(self, cfg):
        super(ConvNet_Simple, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False)
        self.drop1 = Dropout(p=cfg['p'])

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.drop2 = Dropout(p=cfg['p'])

        self.pool = nn.MaxPool2d(2, 2)  # halves spatial size

        self.fc1 = nn.Linear(64 * 7 * 7, 256)  # after pooling twice: 32→16→8
        self.fc2 = nn.Linear(256, cfg['num_classes'])


    def forward(self, x, index=None):

        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = self.drop1(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = self.drop2(x)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

    
    def get_candidates(self, indices):

        return None



In [ ]:
%%writefile student.py
import torch
import torch.nn as nn
import torch.nn.functional as F






class SoftDropout(nn.Module):
    def __init__(self, p, LCMC=True):
        super(SoftDropout, self).__init__()

        self.agg = nn.Conv3d(3, 1, kernel_size=1, bias=True)
        self.p = p
        self.LCMC = LCMC

    def forward(self, x, masks):

        if not self.LCMC:
            return x

        masks = torch.stack(masks, dim=0).unsqueeze(0)
        mask = self.agg(masks)
        mask = F.sigmoid(mask.squeeze(dim=1)) / (1-self.p)

        return x * mask





class ConvNet_s(nn.Module):
    def __init__(self, cfg):
        super(ConvNet_s, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False)  # input: (3,32,32), output: (32,32,32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)  # input: (32,32,32), output: (64,32,32)
        self.pool = nn.MaxPool2d(2, 2)  # halves spatial size

        # self.conv1m = nn.Conv3d(3, 1, kernel_size=(1, 1, 1), bias=True)
        # self.conv2m = nn.Conv3d(3, 1, kernel_size=(1, 1, 1), bias=True)
        self.soft_drop1 = SoftDropout(cfg['p'], cfg['LDMC'])
        self.soft_drop2 = SoftDropout(cfg['p'], cfg['LDMC'])

        self.fc1_ = nn.Linear(64 * 7 * 7, 256)
        self.fc2_ = nn.Linear(256, cfg['num_classes'])

        self.fc3 = nn.Linear(256, 64)
        self.fc4 = nn.Linear(64, 4)
        self.fc5 = nn.Linear(12, 12)
        self.fc6 = nn.Linear(12, 12)

        self.fc7 = nn.Linear(12, 1)


    def forward(self, x, masks):

        x = F.relu(self.conv1(x))
        x = self.pool(x)
        # mask1 = torch.stack(masks[0], dim=0).unsqueeze(0)
        # mask1 = F.sigmoid(self.conv1m(mask1).squeeze(dim=1))
        x = self.soft_drop1(x, masks[0])


        x = F.relu(self.conv2(x))
        x = self.pool(x)
        # mask2 = torch.stack(masks[1], dim=0).unsqueeze(0)
        # mask2 = F.sigmoid(self.conv1m(mask2).squeeze(dim=1))
        x = self.soft_drop1(x, masks[1])

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1_(x))
        cls = self.fc2_(x)

        x = F.relu(self.fc3(x))

        x = self.fc4(x)

        x = F.silu(torch.cat((x, cls), dim=-1))


        x = F.relu(self.fc5(x))
        x = F.relu(self.fc6(x))

        unc = F.softplus(self.fc7(x))

        return cls, unc




In [ ]:
%%writefile evaluation.py
import torch.nn.functional as F
import numpy as np
from torch import nn
import torch
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

def mc_cal(model, dataloader, allowed_indices, T=32, M=10 , R=3, num_classes=10):
    model.train()  # Keep dropout active
    device = next(model.parameters()).device

    # Initialize accumulators
    class_correct = np.zeros(num_classes)
    class_counts = np.zeros(num_classes)
    class_aleatoric = np.zeros(num_classes)
    class_total = np.zeros(num_classes)

    for data, target in dataloader:
        data = data.to(device)
        batch_size = data.size(0)

        # Monte Carlo samples
        candidates = torch.randperm(M)[:R].to(device)
        probs_T = []

        for _ in range(T):
            index = candidates[torch.randint(0, R, [len(target)])]
            out = model(data, index)
            probs = F.softmax(out, dim=1)
            probs_T.append(probs.unsqueeze(0))  # [1, batch, num_classes]
        probs_T = torch.cat(probs_T, dim=0)  # [T, batch, num_classes]

        probs_T_s = probs_T / probs_T.sum(dim=-1, keepdim=True)
        mean_probs_s = probs_T_s.mean(dim=0)

        # Mean predictive distribution
        mean_probs = probs_T.mean(dim=0)  # [batch, num_classes]

        # --- Entropies ---
        # Aleatoric (expected entropy)

        aleatoric_all = -(probs_T_s.clamp(.00001) * probs_T_s.clamp(.00001).log2()).sum(dim=2)
        aleatoric = aleatoric_all.mean(dim=0)


        # Total (predictive) entropy
        total_ent = -(mean_probs_s.clamp(.00001) * mean_probs_s.clamp(.00001).log2()).sum(dim=1)  # [batch]
        # Epistemic = total - aleatoric
        epistemic = total_ent - aleatoric

        preds = mean_probs.argmax(dim=1)

        for i in range(len(target)):
            cls = target[i].item()
            
            class_counts[cls] += 1
            class_aleatoric[cls] += aleatoric[i].item()
            class_total[cls] += total_ent[i].item()
            class_correct[cls] += (allowed_indices[preds[i]] == target[i]).item()

    # Compute averages
    avg_acc = class_correct / np.maximum(class_counts, 1)
    avg_aleatoric = class_aleatoric / np.maximum(class_counts, 1)
    avg_total = class_total / np.maximum(class_counts, 1)
    avg_epistemic = avg_total - avg_aleatoric

    return {
        "accuracy_per_class": avg_acc,
        "aleatoric_per_class": avg_aleatoric,
        "total_per_class": avg_total,
        "epistemic_per_class": avg_epistemic,
    }



def mc_dropout_cal_sample(model, batch, T=20, M=10, R=3):
    model.train()  # keep dropout active

    device = next(model.parameters()).device

    probs_T = []

    batch_size = batch.size(0)
    candidates = torch.randperm(M)[:R].to(device)
    for _ in range(T):
        index = candidates[torch.randint(0, R, [batch_size])]
        out = model(batch, index)
        probs = F.softmax(out, dim=1)
        probs_T.append(probs.unsqueeze(0))  # [1, batch, num_classes]
    probs_T = torch.cat(probs_T, dim=0)  # [T, batch, num_classes]

    masks = model.get_candidates(candidates)

    # Mean predictive distribution
    mean_probs = probs_T.mean(dim=0)  # [batch, num_classes]

    aleatoric_all = -(probs_T.clamp(.001) * probs_T.clamp(.001).log2()).sum(dim=2)
    aleatoric = aleatoric_all.mean(dim=0)


    total_ent = -(mean_probs.clamp(.00001) * mean_probs.clamp(.00001).log2()).sum(dim=-1)  # [batch]
    epistemic = total_ent - aleatoric
    # total_ent_c = get_uncertainty_class(total_ent).to(probs_T.device)
    # aleatoric = get_uncertainty_class(aleatoric).to(probs_T.device)
    # epistemic = get_uncertainty_class(epistemic).to(probs_T.device)


    return {
        "preds": mean_probs,
        "aleatoric": aleatoric,
        "epistemic": epistemic,
        "total": total_ent,
        "total-val": total_ent,
        "masks": masks
    }



def get_uncertainty_class(pred_entropy):
    boarders = [torch.log2(torch.tensor(1.2)), torch.log2(torch.tensor(1.8)), torch.log2(torch.tensor(3)), torch.log2(torch.tensor(5))]

    uncertainty = -torch.ones((len(pred_entropy), 4), dtype=torch.float)
    for j in range(len(pred_entropy)):
        dis = torch.tensor([(pred_entropy[j] - boarders[i]) for i in range(len(boarders))])
        uncertainty[j, :] = dis > 0

    return uncertainty



@torch.no_grad()
def evaluate_student_per_class(student, teacher, test_loader, device, T=20, num_classes=10, M=10, R=3, known_unknown=0, unknown_unknown=1):
    student.eval()
    teacher.train()

    allowed_indices = []
    for i in range(num_classes):
        if i not in [known_unknown, unknown_unknown]:
            allowed_indices.append(i)


    # Initialize accumulators
    class_correct = np.zeros(num_classes)
    class_total = np.zeros(num_classes)
    class_aleatoric = np.zeros(num_classes)
    class_aleatoric_avg = np.zeros(num_classes)
    class_total_ent = np.zeros(num_classes)
    class_kl_loss = np.zeros(num_classes)
    
    loss_func = nn.KLDivLoss(reduction='batchmean')

    all_unc = []
    all_labels = []

    for data, target in test_loader:
        data, target = data.to(device), target.to(device)

        # --- Teacher MC Dropout outputs ---
        detail = mc_dropout_cal_sample(teacher, data, T=T)
        # teacher_preds = detail["preds"]
        # teacher_total = torch.sum(detail["aleatoric"], dim=-1, dtype=torch.long)
        teacher_total = detail["aleatoric"]


        cnadidates = torch.randperm(M)[:R].to(device)
        masks = teacher.get_candidates(cnadidates)

        student_output, student_unc = student(data, masks)
        # probs = F.sigmoid(student_unc)
        # student_unc = torch.sum(probs > .5, dim=1)
        preds = student_output.argmax(dim=1)

        student_pred = F.log_softmax(student_output, dim=1)


        for i in range(len(target)):
            cls = target[i].item()
            
            preds[i] = allowed_indices[preds[i]]
            
            class_total[cls] += 1
            class_correct[cls] += (preds[i] == target[i]).item()
            # print(teacher_total)
            class_aleatoric[cls] += np.power(teacher_total[i].item() - student_unc[i].item(), 2)
            class_aleatoric_avg[cls] += student_unc[i].item()
            class_total_ent[cls] += teacher_total[i].item()
            class_kl_loss[cls] += loss_func(student_pred[i], detail['preds'][i])

            if target[i].item()==unknown_unknown:
                all_unc.append(student_unc[i].item())
                all_labels.append(teacher_total[i].item())

    # Compute averages
    acc_per_class = class_correct / np.maximum(class_total, 1)
    aleatoric_per_class = np.sqrt(class_aleatoric / np.maximum(class_total, 1))
    total_per_class = class_total_ent / np.maximum(class_total, 1)
    aleatoric_avg_per_class = class_aleatoric_avg / np.maximum(class_total, 1)
    class_kl_loss = class_kl_loss / np.maximum(class_total, 1)


    # cm = confusion_matrix(all_labels, all_unc)
    # fig, ax = plt.subplots()
    # im = ax.imshow(cm, cmap='Blues')
    # ax.set_xlabel("Predicted")
    # ax.set_ylabel("True")
    # ax.set_title("Confusion Matrix")
    # plt.colorbar(im)
    # plt.savefig("confusion_matrix.png")



    # Return dictionary
    return {
        "accuracy_per_class": acc_per_class,
        "aleatoric_per_class": aleatoric_per_class,
        "total_per_class": total_per_class,
        "aleatoric_avg_per_class": aleatoric_avg_per_class,
        "class_kl_loss": class_kl_loss
    }


In [ ]:
%%writefile main.py
from evaluation import mc_dropout_cal_sample, evaluate_student_per_class
from teacher import ConvNet, ConvNet_Simple
from teacher_trainer import train_teacher
from student import ConvNet_s
import torch
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms
from torch import nn
import torch.nn.functional as F
from tqdm import tqdm


def initialize_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)



transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = {
    'num_classes': 8, 'p':0.5, 'rw':.4, 'M':10, 'T':128, 'R':3,
    'device': device,
    'LDMC': True
}

batch_size = 32
learning_rate = 0.001
num_epochs = 40


known_unknown, unknown_unknown = 0, 9


for _ in range(4):
    teacher = ConvNet(config).to(device)
    # teacher = ConvNet_Simple(config).to(device)
    teacher.apply(initialize_weights)
    
    # teacher.load_state_dict(torch.load("model/teacher_model.pth"))
    train_teacher(teacher, config, known_unknown=known_unknown, unknown_unknown=unknown_unknown)

    student = ConvNet_s(config).to(device)
        
    
    def filter_digits(dataset, exclude):
        keep_indices = [i for i, (_, label) in enumerate(dataset) if label not in exclude]
        return Subset(dataset, keep_indices)
    
    
    train_dataset = filter_digits(train_dataset, exclude=[unknown_unknown])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    optimizer = torch.optim.Adam(student.parameters(), lr=learning_rate)
    
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.75)
    
    
    criterion_unc = F.binary_cross_entropy_with_logits
    criterion_cls = nn.KLDivLoss(reduction='batchmean')
    

    # initialize student
    student.apply(initialize_weights)
    state_dict = teacher.state_dict()
    student.load_state_dict(state_dict, strict=False)
    
    # for name, param in student.named_children():
    #     if name not in ["fc1_", "fc2_", "fc3", "fc4", "fc5", "fc6",  "soft_drop1", "soft_drop2"]:
    #         param.requires_grad = False
    
    # student.load_state_dict(torch.load("./model/student_model-1.pth"))
    
    gamma = .75
    
    for epoch in range(num_epochs):
        student.train()
        total_loss = 0
    
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
    
            with torch.no_grad():
                detail = mc_dropout_cal_sample(teacher, data, T=config['T'])
    
            optimizer.zero_grad()
            output, unc = student(data, detail['masks'])
            output = F.log_softmax(output, dim=-1)
            loss_cls = criterion_cls(output, detail['preds'])
            loss_unc = F.mse_loss(unc.squeeze(), detail["aleatoric"])
    
            loss = loss_cls + gamma * loss_unc
    
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
        scheduler.step()
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}")
    
        if (epoch+1) % 10 == 0:
            torch.save(student.state_dict(), "./model/student_model-1.pth")
            print(evaluate_student_per_class(student, teacher, train_loader, device, T=config['T'], num_classes=10,
                                            known_unknown=known_unknown, unknown_unknown=unknown_unknown,
                                            M=['M'], R=['R']))
    
        if (epoch+1) % 10 == 0:
            print(evaluate_student_per_class(student, teacher, test_loader, device, T=['T'], num_classes=10,
                                            known_unknown=known_unknown, unknown_unknown=unknown_unknown,
                                            M=['M'], R=['R']))
    print("***********----------------")


In [ ]:
!python main.py